In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import os
from sklearn.model_selection import train_test_split
from scipy import stats
from scipy.stats import epps_singleton_2samp
from PIL import Image
from train_test_helper import optimize_split, get_cell_counts
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)


In [ ]:
# Getting Annotations and Exploring Blood Cell counts. This is to find out if the dataset is balanced or not.
anno_dir = '../data/BCCD/Annotations/'
xml_data, rbc_counts, wbc_counts, platelet_counts = get_cell_counts(anno_dir)
rbc_sum = np.sum(rbc_counts)
wbc_sum = np.sum(wbc_counts)
platelet_sum = np.sum(platelet_counts)

print('COUNTS')
print(f'Total RBC count: {rbc_sum}')
print(f'Total WBC count: {wbc_sum}')
print(f'Total Platelet count: {platelet_sum}')

# Visualizing the TOTAL counts of each blood cell type using a bar plot
cell_types = ['RBC', 'WBC', 'Platelets']
counts = [rbc_sum, wbc_sum, platelet_sum]
plt.figure(figsize=(8, 6))
sns.barplot(x=cell_types, y=counts)
plt.title('Counts of Blood Cell Types in BCCD Dataset')
print("Clearly there is a class imbalance in the dataset. RBCs are the most abundant, followed by WBCs and Platelets.")
plt.show()

#However, this does make sense...there should be more RBCs than WBCs and Platelets in a blood smear. A better
# way to look at the data is to see how many images contain each type of blood cell. This will give us a better idea of how
# balanced the dataset is in terms of the number of images containing each type of blood cell.

images_with_rbc = 0
images_with_wbc = 0
images_with_platelets = 0
images_with_no_rbc = 0
images_with_no_wbc = 0
images_with_no_platelets = 0
for data in xml_data:
    for key in data.keys():
        if data[key]['counts']['RBC'] > 0:
            images_with_rbc += 1
        if data[key]['counts']['WBC'] > 0:
            images_with_wbc += 1
        if data[key]['counts']['Platelets'] > 0:
            images_with_platelets += 1
        if data[key]['counts']['RBC'] == 0:
            images_with_no_rbc += 1
        if data[key]['counts']['WBC'] == 0:
            images_with_no_wbc += 1
        if data[key]['counts']['Platelets'] == 0:
            images_with_no_platelets += 1

print('IMAGES CONTAINING EACH BLOOD CELL TYPE')
print(f'Images with RBCs: {images_with_rbc}')
print(f'Images with WBCs: {images_with_wbc}')
print(f'Images with Platelets: {images_with_platelets}')
print(f'Images with no RBCs: {images_with_no_rbc}')
print(f'Images with no WBCs: {images_with_no_wbc}')
print(f'Images with no Platelets: {images_with_no_platelets}')

# in this piece of code, I plot the distribution of RBC, WBC, and Platelet counts per image respectively.
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
sns.histplot(rbc_counts, bins=10)
plt.title('Distribution of RBC Counts per Image')
plt.subplot(1, 3, 2)
sns.histplot(wbc_counts, bins=10)
plt.title('Distribution of WBC Counts per Image')
plt.subplot(1, 3, 3)
sns.histplot(platelet_counts, bins=10)
plt.title('Distribution of Platelet Counts per Image')

In [ ]:
# EDA on cell sizes


jpg_dir = '../data/BCCD/JPEGImages/'
jpgs_in_jpg_dir = sorted([f for f in os.listdir(jpg_dir) if f.endswith(".jpg")])
sizes_dict = {}
ix = 0
for entry in xml_data:
    blood_image_name = list(entry.keys())[0]
    sizes_dict[blood_image_name] = {"RBC":[], "WBC":[], "Platelets":[]}
    filename = blood_image_name+".jpg"
    filename = os.path.join(jpg_dir, filename)

    
    if ix == 3:
        # Load the image
        image = Image.open(filename)
        # Show it
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.title(blood_image_name)

    for key, item in entry[blood_image_name].items():
        try:
            cell_type = entry[blood_image_name][key]['name']
            xmin = entry[blood_image_name][key]['xmin']
            xmax = entry[blood_image_name][key]['xmax']
            ymin = entry[blood_image_name][key]['ymin']
            ymax = entry[blood_image_name][key]['ymax']
            pt1 = (xmin, ymin)
            pt2 = (xmin, ymax)
            pt3 = (xmax, ymin)
            pt4 = (xmax, ymax)
            area = (xmax - xmin) * (ymax - ymin)/(image.width*image.height)
            sizes_dict[blood_image_name][cell_type].append(area)
            if ix == 3:
                if cell_type == "RBC":
                    color = 'red'
                elif cell_type == "WBC":
                    color = "blue"
                else:
                    color = "black"
                plt.plot([xmin, xmax],[ymin, ymin], color=color)
                plt.plot([xmax, xmax],[ymin, ymax], color=color)
                plt.plot([xmin, xmax],[ymax, ymax], color=color)
                plt.plot([xmin, xmin],[ymax, ymin], color=color)
            
                plt.text(xmin, ymin - 5, f'{cell_type} Area: {area*1000:.2f}', color=color, fontsize=8) #mult by 1000 just for readibility
        except KeyError:
            pass
    ix+=1
i = 0
all_rbc_areas = []
all_wbc_areas = []
all_platelet_areas = []

for key, item in sizes_dict.items():
    rbc_areas = item['RBC']
    rbc_areas = [i*1000 for i in rbc_areas] # mult by 1000 just for readibility
    wbc_areas = item['WBC']
    wbc_areas = [i*1000 for i in wbc_areas] # mult by 1000 just for readibility
    platelet_areas = item['Platelets']
    platelet_areas = [i*1000 for i in platelet_areas] # mult by 1000 just for readibility
    all_rbc_areas.extend(rbc_areas)
    all_wbc_areas.extend(wbc_areas)
    all_platelet_areas.extend(platelet_areas)


# Plot the distribution of cell areas by type
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(np.asarray(all_rbc_areas), bins=30, kde=True, color='red', stat='percent')
plt.title('RBC Area Distribution')
plt.xlabel('Relative cell area')
plt.ylabel('Count')

plt.subplot(1, 3, 2)
sns.histplot(np.asarray(all_wbc_areas), bins=30, kde=True, color='blue', stat = 'percent')
plt.title('WBC Area Distribution')
plt.xlabel('Relative cell area')
plt.ylabel('Count')

plt.subplot(1, 3, 3)
sns.histplot(np.asarray(all_platelet_areas), bins=30, kde=True, color='black', stat='percent')
plt.title('Platelet Area Distribution')
plt.xlabel('Relative cell area')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# This cell is a divergence from EDA. This takes what I have learned above from raw cell counts
# and sets up an optimized train/test splitting routine that should have the train/test splits
# have very similar distributions of cell counts per image across each cell type.
# This preserves class integrity across train/test splits (e.g., a truly random split might have few
# to no platelets in test and most of them in train).
from train_test_helper import optimize_split

train_idx, test_idx = train_test_split(np.arange(len(xml_data)), test_size=0.3)

train_idx = train_idx.tolist()
test_idx = test_idx.tolist()

train_idx, test_idx, all_train_losses_rb, all_test_losses_rb, all_train_losses_wb, all_test_losses_wb, all_train_losses_pl,\
all_test_losses_pl, all_train_losses, all_test_losses = optimize_split(rbc_counts, wbc_counts, platelet_counts, train_idx, test_idx, n_iterations=15000, verbose=True)


print()
_, p = epps_singleton_2samp(rbc_counts, np.array(rbc_counts)[train_idx])
print(f"P-value for RBC train distribution vs original distribution: {p:4f}")
_, p = epps_singleton_2samp(rbc_counts, np.array(rbc_counts)[test_idx])
print(f"P-value for RBC test distribution vs original distribution: {p:4f}")
try:
    _, p = epps_singleton_2samp(wbc_counts, np.array(wbc_counts)[train_idx])
    print(f"P-value for WBC train distribution vs original distribution: {p:4f}")
except np.linalg.LinAlgError:
    print('p-value not found for train WBC distributions ')

try:
    _, p = epps_singleton_2samp(wbc_counts, np.array(wbc_counts)[test_idx])
    print(f"P-value for WBC test distribution vs original distribution: {p:4f}")
except np.linalg.LinAlgError:
    print('p-value not found for test WBC distributions ')

try:
    _, p = epps_singleton_2samp(platelet_counts, np.array(platelet_counts)[train_idx])
    print(f"P-value for Platelet train distribution vs original distribution: {p:.4f}")
except np.linalg.LinAlgError:
    print('p-value not found for train Platelet distributions ')

try:
    _, p = epps_singleton_2samp(platelet_counts, np.array(platelet_counts)[test_idx])
    print(f"P-value for Platelet test distribution vs original distribution: {p:.4f}")
except np.linalg.LinAlgError:
    print('p-value not found for test Platelet distributions ')

plt.figure(figsize=(10, 6))
plt.plot(all_train_losses, label='Train Loss')
plt.plot(all_test_losses, label='Test Loss')
plt.plot(all_train_losses_rb, label = 'Train RB Loss')
plt.plot(all_test_losses_rb, label = "Test RB Loss")
plt.plot(all_train_losses_wb, label='Train WBC Loss')
plt.plot(all_test_losses_wb, label='Test WBC Loss')
plt.plot(all_train_losses_pl, label='Train Platelet Loss')
plt.plot(all_test_losses_pl, label='Test Platelet Loss')
plt.title('Loss over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Wasserstein Distance')
plt.legend()
plt.show()

# in this piece of code, I plot the distribution of RBC, WBC, and Platelet counts per image respectively.
plt.figure()
fig, axes = plt.subplots(1, 3, figsize=(12,6))

sns.histplot(rbc_counts, bins=20, label = 'Original', stat='density', ax = axes[0])
sns.histplot(np.array(rbc_counts)[train_idx], bins = 20, label= 'Train', stat='density', ax = axes[0])
sns.histplot(np.array(rbc_counts)[test_idx], bins=20, label='Test', stat='density', ax = axes[0])
axes[0].set_title("RBC")
axes[0].legend()

plt.subplot(1,3,2)
sns.histplot(wbc_counts, bins=10, label='Original', stat='density', ax=axes[1])
sns.histplot(np.array(wbc_counts)[train_idx], bins=10, label='Train', stat='density', ax=axes[1])
sns.histplot(np.array(wbc_counts)[test_idx], bins=10, label='Test', stat='density', ax=axes[1])
axes[1].set_title("WBC")
axes[1].legend()

plt.subplot(1,3,3)
sns.histplot(platelet_counts, bins=10, label = 'Original', stat='density', ax=axes[2])
sns.histplot(np.array(platelet_counts)[train_idx], bins=10, label = "Train", stat='density', ax=axes[2])
sns.histplot(np.array(platelet_counts)[test_idx], bins=10, label='Test', stat='density', ax=axes[2])
axes[2].set_title("Platelet")
axes[2].legend()



In [ ]:
# simple 50/50 split to init val/test split
# val_idx = test_idx[:int(len(test_idx)/2)]
# test_idx_2 = test_idx[int(len(test_idx)/2):]
test_idx_2, val_idx = train_test_split(test_idx, test_size=0.5)

val_idx = list(val_idx)
test_idx_2 = list(test_idx_2)


test_idx_2, val_idx, all_test_losses_rb, all_val_losses_rb, all_test_losses_wb, all_val_losses_wb, all_test_losses_pl,\
all_val_losses_pl, all_test_losses, all_val_losses = optimize_split(rbc_counts, wbc_counts, platelet_counts, test_idx_2, val_idx,
                                                                    n_iterations=2000 ,verbose=True)


print()
_, p = epps_singleton_2samp(rbc_counts, np.array(rbc_counts)[test_idx_2])
print(f"P-value for RBC test distribution vs original distribution: {p:4f}")
_, p = epps_singleton_2samp(rbc_counts, np.array(rbc_counts)[val_idx])
print(f"P-value for RBC val distribution vs original distribution: {p:4f}")
try:
    _, p = epps_singleton_2samp(wbc_counts, np.array(wbc_counts)[test_idx_2])
    print(f"P-value for WBC test distribution vs original distribution: {p:4f}")
except np.linalg.LinAlgError:
    print('p-value not found for test WBC distributions ')

try:
    _, p = epps_singleton_2samp(wbc_counts, np.array(wbc_counts)[val_idx])
    print(f"P-value for WBC val distribution vs original distribution: {p:4f}")
except np.linalg.LinAlgError:
    print('p-value not found for val WBC distributions ')

try:
    _, p = epps_singleton_2samp(platelet_counts, np.array(platelet_counts)[test_idx_2])
    print(f"P-value for Platelet test distribution vs original distribution: {p:.4f}")
except np.linalg.LinAlgError:
    print('p-value not found for test Platelet distributions ')

try:
    _, p = epps_singleton_2samp(platelet_counts, np.array(platelet_counts)[val_idx])
    print(f"P-value for Platelet val distribution vs original distribution: {p:.4f}")
except np.linalg.LinAlgError:
    print('p-value not found for val Platelet distributions ')

plt.figure(figsize=(10, 6))
plt.plot(all_test_losses, label='Test Loss')
plt.plot(all_val_losses, label='Val Loss')
plt.plot(all_test_losses_rb, label = 'Test RB Loss')
plt.plot(all_val_losses_rb, label = "Val RB Loss")
plt.plot(all_test_losses_wb, label='Test WBC Loss')
plt.plot(all_val_losses_wb, label='Val WBC Loss')
plt.plot(all_test_losses_pl, label='Test Platelet Loss')
plt.plot(all_val_losses_pl, label='Val Platelet Loss')
plt.title('Loss over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Wasserstein Distance')
plt.legend()
plt.show()

# in this piece of code, I plot the distribution of RBC, WBC, and Platelet counts per image respectively.
plt.figure()
fig, axes = plt.subplots(1, 3, figsize=(12,6))

sns.histplot(rbc_counts, bins=20, label = 'Original', stat='density', ax = axes[0])
sns.histplot(np.array(rbc_counts)[test_idx_2], bins = 20, label= 'Test', stat='density', ax = axes[0])
sns.histplot(np.array(rbc_counts)[val_idx], bins=20, label='Val', stat='density', ax = axes[0])
axes[0].set_title("RBC")
axes[0].legend()

plt.subplot(1,3,2)
sns.histplot(wbc_counts, bins=10, label='Original', stat='density', ax=axes[1])
sns.histplot(np.array(wbc_counts)[test_idx_2], bins=10, label='Test', stat='density', ax=axes[1])
sns.histplot(np.array(wbc_counts)[val_idx], bins=10, label='Val', stat='density', ax=axes[1])
axes[1].set_title("WBC")
axes[1].legend()

plt.subplot(1,3,3)
sns.histplot(platelet_counts, bins=10, label = 'Original', stat='density', ax=axes[2])
sns.histplot(np.array(platelet_counts)[test_idx_2], bins=10, label = "Test", stat='density', ax=axes[2])
sns.histplot(np.array(platelet_counts)[val_idx], bins=10, label='Val', stat='density', ax=axes[2])
axes[2].set_title("Platelet")
axes[2].legend()